# Apply SV to WT Chromosome (and motifs)
- This notebook applies structural variations to matrix representations of a chromatin locus, either directly to a contact map or to an effective potential matrix
- Can also be used with an extrusion model to apply SVs to motif tracks

### Under the `Main' heading, you'll find the variables you need to change to run this notebook on your data

In [1]:
import numpy as np
from numpy.typing import NDArray
from typing import Optional
import matplotlib.pyplot as plt

# Helper Functions

In [2]:
def _open_lambdas(file):
    return(np.genfromtxt(file, delimiter=',',dtype=float, skip_header=1,filling_values=0))

In [ ]:
def _open_contact_map(file: str) -> NDArray[np.float64]:
    """Open a dense matrix from .npy or text/dense formats."""
    if file.lower().endswith(".npy"):
        return np.load(file)

    try:
        return np.loadtxt(file)
    except ValueError:
        return np.loadtxt(file)

In [5]:
# Helper functions for array handling

def _as_numeric_square_array(mat: NDArray[np.number], name: str = "mat") -> NDArray[np.number]:
    """Validate a square numeric matrix input and return it as a NumPy array."""
    if not isinstance(mat, np.ndarray):
        raise TypeError(f"{name} must be a NumPy array")
    if mat.ndim != 2:
        raise ValueError(f"{name} must be a 2D matrix")
    if mat.shape[0] != mat.shape[1]:
        raise ValueError(f"{name} must be square")
    if mat.shape[0] == 0:
        raise ValueError(f"{name} must be non-empty")
    if not np.issubdtype(mat.dtype, np.number):
        raise TypeError(f"{name} must contain numeric values")
    return mat

def _as_numeric_vector(vec: NDArray[np.number], name: str = "vec") -> NDArray[np.float64]:
    """Validate a numeric vector input and return it as a NumPy float array."""
    if not isinstance(vec, np.ndarray):
        raise TypeError(f"{name} must be a NumPy array")
    if vec.ndim != 1:
        raise ValueError(f"{name} must be a 1D vector")
    if vec.shape[0] == 0:
        raise ValueError(f"{name} must be non-empty")
    if not np.issubdtype(vec.dtype, np.number):
        raise TypeError(f"{name} must contain numeric values")
    return vec



In [ ]:
# Helper functions for SV implementation

# add storage diagonal averages from some 'IC curve'
def _add_IC(mat: NDArray[np.number], avgs: NDArray[np.number]) -> NDArray[np.float64]:
    """Add an ideal chromosome (IC) diagonal-average signal to a square matrix.

    Parameters
    ----------
    mat : np.ndarray
        Square 2D numeric matrix to receive the IC diagonal averages.
    avgs : np.ndarray
        1D numeric vector of diagonal averages. Its length must be at least the
        matrix width so every diagonal offset has a value.

    Returns
    -------
    np.ndarray
        Symmetric matrix with `avgs[j - i]` added to each upper-triangle value.

    Raises
    ------
    TypeError
        If `mat` or `avgs` are not supported numeric inputs.
    ValueError
        If `mat` is not a non-empty square 2D matrix, or if `avgs` is too short
        or contains non-finite values.
    """
    mat = _as_numeric_square_array(mat)
    avgs = _as_numeric_vector(avgs, name="avgs")
    N = mat.shape[0]
    n = avgs.shape[0]

    mat_with_IC = np.zeros((N, N), dtype=float)


    for i in range(N):
        for j in range(i, min(N,n+i)):
            mat_with_IC[i,j] = mat[i,j] + avgs[j - i]

    mat_with_IC_bottom = mat_with_IC.copy().T
    np.fill_diagonal(mat_with_IC_bottom, 0)
    return mat_with_IC + mat_with_IC_bottom


# this removes the diagonal averages ('ideal chromosome potential') from a matrix and returns the IC curve as a vector
def _remove_IC(mat: NDArray[np.number]) -> tuple[NDArray[np.float64], NDArray[np.float64]]:
    """Remove the ideal chromosome (IC) diagonal-average signal from a square matrix.

    Parameters
    ----------
    mat : np.ndarray
        Square 2D numeric matrix. NaN values are ignored when computing each
        upper-triangle diagonal average.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        A tuple containing the IC-adjusted symmetric matrix and the vector of
        diagonal averages that were subtracted.

    Raises
    ------
    TypeError
        If `mat` is not a NumPy array or does not contain numeric values.
    ValueError
        If `mat` is not a non-empty square 2D matrix, or if any diagonal has
        no finite values from which to compute an average.
    """
    mat = _as_numeric_square_array(mat)

    N = mat.shape[0]

    # get the average val of each diagonal of the upper triangle
    avgs = np.array(
        [np.nanmean(np.diag(mat, k=offset)) for offset in range(N)],
        dtype=float,
    )
    if not np.isfinite(avgs).all():
        raise ValueError("each diagonal must have a finite average")

    # buffer around 0
    avgs[avgs == 0] = 10**-4

    # subtract diagonal average from each element
    mat_no_IC = np.zeros((N, N), dtype=float)
    for i in range(N):
        for j in range(i, N):
            mat_no_IC[i,j] = mat[i,j] - avgs[j - i]

    mat_no_IC_bottom = mat_no_IC.copy().T
    np.fill_diagonal(mat_no_IC_bottom, 0)
    out = mat_no_IC + mat_no_IC_bottom

    return out, avgs


In [ ]:
# Implement SVs on matrices

# implement deletion
def _apply_deletion(
    mat: NDArray[np.number],
    start: int,
    end: int,
    adj_IC: bool = False,
) -> NDArray[np.float64]:
    """Return a matrix after deleting rows and columns from `start` to `end`.

    Parameters
    ----------
    mat : np.ndarray
        Square 2D numeric matrix to edit.
    start : int
        Inclusive start index of the deleted interval.
    end : int
        Exclusive end index of the deleted interval.
    adj_IC : bool, default False
        If True, remove the IC diagonal averages before deletion and add them
        back to the deleted matrix afterward.

    Returns
    -------
    np.ndarray
        Matrix with the selected rows and columns removed.

    Raises
    ------
    TypeError
        If inputs have the wrong type.
    ValueError
        If the matrix is invalid or the deletion interval is out of bounds.
    """
    mat = _as_numeric_square_array(mat)
    if not isinstance(start, int) or isinstance(start, bool):
        raise TypeError("start must be an integer")
    if not isinstance(end, int) or isinstance(end, bool):
        raise TypeError("end must be an integer")
    if not isinstance(adj_IC, bool):
        raise TypeError("adj_IC must be a boolean")

    N = mat.shape[0]
    if start < 0 or end > N:
        raise ValueError("start and end must be within matrix bounds")
    if start >= end:
        raise ValueError("start must be less than end")
    if end - start >= N:
        raise ValueError("deletion cannot remove the entire matrix")

    if adj_IC:
        temp, avgs = _remove_IC(mat)
    else:
        temp = mat.copy()

    del_row_col = np.r_[0:start, end:N]
    out = temp[np.ix_(del_row_col, del_row_col)]

    if adj_IC:
        out = _add_IC(out, avgs)

    return out


# implement inversion
def _apply_inversion(
    mat: NDArray[np.number],
    start: int,
    end: int,
    adj_IC: bool = False,
) -> NDArray[np.float64]:
    """Return a matrix with rows and columns in `[start, end)` reversed.

    If `adj_IC` is True, the IC diagonal averages are removed before inversion
    and added back afterward.
    """
    mat = _as_numeric_square_array(mat)
    if not isinstance(start, int) or isinstance(start, bool):
        raise TypeError("start must be an integer")
    if not isinstance(end, int) or isinstance(end, bool):
        raise TypeError("end must be an integer")
    if not isinstance(adj_IC, bool):
        raise TypeError("adj_IC must be a boolean")

    N = mat.shape[0]
    if start < 0 or end > N:
        raise ValueError("start and end must be within matrix bounds")
    if start >= end:
        raise ValueError("start must be less than end")

    if adj_IC:
        temp, diag_avgs = _remove_IC(mat)
    else:
        temp = mat

    inv_idx = np.r_[0:start, np.arange(end - 1, start - 1, -1), end:N]
    out = temp[np.ix_(inv_idx, inv_idx)]

    if adj_IC:
        out = _add_IC(out, diag_avgs)

    return out

# implement duplication
# dup_gen: if this is turned on, then the inter-duplicate contacts will be left blank
    # (if adj_IC is also on, then they will be filled with diagonal averages)
def _apply_duplication(
    mat: NDArray[np.number],
    dup_start: int,
    dup_end: int,
    adj_IC: bool = True,
    dup_gen: bool = False,
) -> NDArray[np.float64]:
    """Return a matrix with `[dup_start, dup_end)` duplicated after `dup_end`.

    If `dup_gen` is True, contacts involving the inserted duplicate are left
    blank before IC is added back.
    """
    mat = _as_numeric_square_array(mat)
    if not isinstance(dup_start, int) or isinstance(dup_start, bool):
        raise TypeError("dup_start must be an integer")
    if not isinstance(dup_end, int) or isinstance(dup_end, bool):
        raise TypeError("dup_end must be an integer")
    if not isinstance(adj_IC, bool):
        raise TypeError("adj_IC must be a boolean")
    if not isinstance(dup_gen, bool):
        raise TypeError("dup_gen must be a boolean")

    N = mat.shape[0]
    if dup_start < 0 or dup_end > N:
        raise ValueError("dup_start and dup_end must be within matrix bounds")
    if dup_start >= dup_end:
        raise ValueError("dup_start must be less than dup_end")

    if adj_IC:
        temp, diag_avgs = _remove_IC(mat)
    else:
        temp = mat.copy()

    dup_idx = np.r_[0:dup_end, dup_start:dup_end, dup_end:N]
    out = temp[np.ix_(dup_idx, dup_idx)]

    if dup_gen:
        dup_len = dup_end - dup_start
        inserted = np.s_[dup_end:dup_end + dup_len]
        out[inserted, :] = 0
        out[:, inserted] = 0

    if adj_IC:
        out = _add_IC(out, diag_avgs)

    return out


In [ ]:
# duplicate, delete, or invert motif tracks

def _delete_motifs(
    ms: NDArray[np.number],
    start: int,
    end: int,
) -> NDArray[np.float64]:
    """Return `ms` with the motif interval `[start, end)` removed."""
    # Validate motif track inputs before deleting the selected interval.
    ms = _as_numeric_vector(ms, name="ms")
    if not isinstance(start, int) or isinstance(start, bool):
        raise TypeError("start must be an integer")
    if not isinstance(end, int) or isinstance(end, bool):
        raise TypeError("end must be an integer")
    N = ms.shape[0]
    if start < 0 or end > N:
        raise ValueError("start and end must be within motif bounds")
    if start >= end:
        raise ValueError("start must be less than end")
    if end - start >= N:
        raise ValueError("deletion cannot remove the entire motif track")
    return np.concatenate((ms[:start], ms[end:]))

def _invert_motifs(
    fs: NDArray[np.number],
    rs: NDArray[np.number],
    start: int,
    end: int,
) -> tuple[NDArray[np.float64], NDArray[np.float64]]:
    """Return forward and reverse motif tracks after inverting `[start, end)`."""
    # Validate paired motif tracks before reversing and swapping strands.
    fs = _as_numeric_vector(fs, name="fs")
    rs = _as_numeric_vector(rs, name="rs")
    if not isinstance(start, int) or isinstance(start, bool):
        raise TypeError("start must be an integer")
    if not isinstance(end, int) or isinstance(end, bool):
        raise TypeError("end must be an integer")
    if fs.shape != rs.shape:
        raise ValueError("fs and rs must have the same shape")
    N = fs.shape[0]
    if start < 0 or end > N:
        raise ValueError("start and end must be within motif bounds")
    if start >= end:
        raise ValueError("start must be less than end")
    return (
        np.concatenate((fs[:start], rs[start:end][::-1], fs[end:])),
        np.concatenate((rs[:start], fs[start:end][::-1], rs[end:])),
    )
  
def _duplicate_motifs(
    ms: NDArray[np.number],
    start: int,
    end: int,
) -> NDArray[np.float64]:
    """Return `ms` with the motif interval `[start, end)` duplicated after `end`."""
    # Validate motif track inputs before duplicating the selected interval.
    ms = _as_numeric_vector(ms, name="ms")
    if not isinstance(start, int) or isinstance(start, bool):
        raise TypeError("start must be an integer")
    if not isinstance(end, int) or isinstance(end, bool):
        raise TypeError("end must be an integer")
    N = ms.shape[0]
    if start < 0 or end > N:
        raise ValueError("start and end must be within motif bounds")
    if start >= end:
        raise ValueError("start must be less than end")
    return np.concatenate((ms[:end], ms[start:end], ms[end:]))

# Apply SV Function

In [ ]:
def apply_SV(
    type: str,
    start: int,
    end: int,
    matrix_path: str,
    matrix_is_epm: bool = True,
    fs_path: Optional[str] = None,
    rs_path: Optional[str] = None,
    apply_to_motifs: bool = False,
    adjust_IC: bool = False,
    dup_gen: bool = False,
) -> NDArray[np.float64] | tuple[NDArray[np.float64], NDArray[np.float64], NDArray[np.float64]]:
    """Apply a duplication, deletion, or inversion to an EPM and optional motif tracks."""
    # Validate SV request inputs, then dispatch to the matrix and motif helpers.
    if not isinstance(type, str):
        raise TypeError("type must be a string")
    if type not in {"dup", "del", "inv"}:
        raise ValueError("type must be one of {'dup', 'del', 'inv'}")
    if not isinstance(start, int) or isinstance(start, bool):
        raise TypeError("start must be an integer")
    if not isinstance(end, int) or isinstance(end, bool):
        raise TypeError("end must be an integer")
    if not isinstance(matrix_path, str):
        raise TypeError("epm_path must be a string")
    if not isinstance(apply_to_motifs, bool):
        raise TypeError("apply_to_motifs must be a boolean")
    if not isinstance(adjust_IC, bool):
        raise TypeError("adjust_IC must be a boolean")
    if not isinstance(dup_gen, bool):
        raise TypeError("dup_gen must be a boolean")

    if matrix_is_epm:
        matrix = _open_lambdas(matrix_path)
    else:
        matrix = _open_contact_map(matrix_path)

    fs = rs = None
    if apply_to_motifs:
        if not isinstance(fs_path, str):
            raise TypeError("fs_path must be a string")
        if not isinstance(rs_path, str):
            raise TypeError("rs_path must be a string")
        fs = np.load(fs_path)
        rs = np.load(rs_path)

    if type == "dup":
        out = _apply_duplication(matrix, start, end, adj_IC=adjust_IC, dup_gen=dup_gen)
        motif_out = (_duplicate_motifs(fs, start, end), _duplicate_motifs(rs, start, end)) if apply_to_motifs else None
    elif type == "del":
        out = _apply_deletion(matrix, start, end, adj_IC=adjust_IC)
        motif_out = (_delete_motifs(fs, start, end), _delete_motifs(rs, start, end)) if apply_to_motifs else None
    else:
        out = _apply_inversion(matrix, start, end, adj_IC=adjust_IC)
        motif_out = _invert_motifs(fs, rs, start, end) if apply_to_motifs else None

    return (out, *motif_out) if apply_to_motifs else out

# Main

In [ ]:
# if loading a HiC map directly, should be .npy, .dense, or .txt
# if loading a potential matrix, should be a .csv with the full inversion header formatting 
matrix_path = 'path/to/potential-matrix-or-hic-map/(.csv if potential matrix, stored in Full Inversion-compatible format)'

# should be of same length as matrix, and both should be .npy
fs_path = 'path/to/fs.npy'
rs_path = 'path/to/rs.npy'

# if matrix is in EPM (.csv with header) format, set to True. If in one of the numpy formats, set to false. 
matrix_is_epm = True


# if you're loading and adjusting motifs as well as the matrix. 
apply_to_motifs = False

# if you'd like to perform ideal chromosome adjustment
adjust_IC = True

# choose from {'dup', 'del', 'inv'}
SV_type = 'del'

# if you're duplicating, can choose dupgen or not
dup_gen = False

# start = start of SV, end = end of SV (positions on matrix representation)
start = 538
end = 706

# Run Functions

In [ ]:
data = apply_SV(type = SV_type,
                start = start,
                end = end,
                matrix_path=matrix_path,
                matrix_is_epm=matrix_is_epm,
                fs_path=fs_path,
                rs_path=rs_path,
                apply_to_motifs=apply_to_motifs,
                adjust_IC=adjust_IC,
                dup_gen=dup_gen,

            )

if apply_to_motifs: SV_matrix,SV_fs,SV_rs = data
else: SV_matrix = data

In [ ]:
plt.imshow(SV_matrix,vmax=.1,vmin=-.2,cmap='Reds')
plt.show()